In [7]:
import numpy as np
import matplotlib.pyplot as plt
import sys
import xarray
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)
from sklearn.ensemble import HistGradientBoostingRegressor
sys.path.append("../src/climate_trends/")

import load as ld
import xarray as xr
import pandas as pd
from statsmodels.tsa.seasonal import STL
import emcee
from statsmodels.tsa.statespace.structural import UnobservedComponents


# Summary
In this notebook I apply two classical ML algorithm formalisms to predict the average temperature for the next months.

# Load the data [for Paris]

In [8]:
## Load the dataset for Paris
ds_p=ld.load_single_file("../data/raw/era5_paris_t2m.nc")
t2m_p = ds_p["t2m"].sel(latitude=48.75,longitude=2.25,method='nearest') -273.15
df = pd.DataFrame({
    "valid_time": pd.to_datetime(t2m_p["valid_time"].values),
    "t2m": t2m_p.values
})
## Convert to panda format
print(type(df["t2m"]),type(df["valid_time"]))

<class 'pandas.Series'> <class 'pandas.Series'>


## Feature Engineering

In [9]:
# ----------------------------
# Time features
# ----------------------------
df["year"] = df["valid_time"].dt.year
df["month"] = df["valid_time"].dt.month

# Long-term trend (months since start)
df["trend"] = np.arange(len(df))

# Seasonal encoding
df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)

# ----------------------------
# Lag Features (Previous 12 months)
# ----------------------------
for lag in range(1, 13):
    df[f"lag_{lag}"] = df["t2m"].shift(lag)


# ------------------------------------------------------------
# Rolling Statistics
# ------------------------------------------------------------

df["rolling_mean_3"] = df["t2m"].shift(1).rolling(3).mean()
df["rolling_mean_6"] = df["t2m"].shift(1).rolling(6).mean()
df["rolling_mean_12"] = df["t2m"].shift(1).rolling(12).mean()

df["rolling_std_3"] = df["t2m"].shift(1).rolling(3).std()
df["rolling_std_6"] = df["t2m"].shift(1).rolling(6).std()
df["rolling_std_12"] = df["t2m"].shift(1).rolling(12).std()



# ----------------------------
# Feature columns
# ----------------------------
#feature_cols = ["trend", "month_cos","month_sin", "rolling_mean_12","rolling_std_12", 
#                "rolling_mean_3","rolling_std_3","rolling_mean_6","rolling_std_6",
#    ] + [f"lag_{lag}" for lag in range(1, 13)]

feature_cols = ["trend", "month_cos","month_sin"
    ] + [f"lag_{lag}" for lag in range(1, 13)]
# Save the last column for forecasting
# Row used for forecasting
forecast_df = df.tail(1).copy()
X_forecast=forecast_df[feature_cols]
# Define the target variable for prediction
df["target"] = df["t2m"].shift(-1)

# ----------------------------
# Remove rows with NAN values
# ----------------------------
df = df.dropna().reset_index(drop=True)

## Define the X and Y of dataset for prediction
X = df[feature_cols]
y = df["target"]

## Test Train Split

In [10]:
# Last 10 years as test set
test_size = 120

X_train = X.iloc[:-test_size]
X_test = X.iloc[-test_size:]

y_train = y.iloc[:-test_size]
y_test = y.iloc[-test_size:]

# Gradient Boost Model

In [11]:

model = HistGradientBoostingRegressor(
    learning_rate=0.05,
    max_depth=6,
    max_iter=500,
    random_state=42,
)

model.fit(X_train, y_train)

# ------------------------------------------------------------
# Prediction
# ------------------------------------------------------------
y_pred=model.predict(X_test)

# ------------------------------------------------------------
# Evaluation
# ------------------------------------------------------------

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("=" * 40)
print("Evaluation Metrics")
print("=" * 40)
print(f"RMSE : {rmse:.3f} °C")
print(f"MAE  : {mae:.3f} °C")
print(f"R²   : {r2:.3f}")



Evaluation Metrics
RMSE : 1.834 °C
MAE  : 1.468 °C
R²   : 0.896


## Get the prediction for Jan,25 (From Gradient Boost)

In [12]:
next_month_prediction = model.predict(X_forecast)[0]

print("\nForecast for next month")
print(f"{next_month_prediction:.2f} °C")


Forecast for next month
4.80 °C


# Random Forest Model

In [16]:
from sklearn.ensemble import RandomForestRegressor

# ------------------------------------------------------------
# Random Forest Model
# ------------------------------------------------------------
model_RF = RandomForestRegressor(
    n_estimators=500,
    max_depth=15,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features='sqrt',
    bootstrap=True,
    random_state=42,
    n_jobs=-1
)

# Train
model_RF.fit(X_train, y_train)

# ------------------------------------------------------------
# Prediction
# ------------------------------------------------------------
y_pred_RF = model_RF.predict(X_test)
# ------------------------------------------------------------
# Evaluation
# ------------------------------------------------------------

rmse = np.sqrt(mean_squared_error(y_test, y_pred_RF))
mae = mean_absolute_error(y_test, y_pred_RF)
r2 = r2_score(y_test, y_pred_RF)

print("=" * 40)
print("Evaluation Metrics")
print("=" * 40)
print(f"RMSE : {rmse:.3f} °C")
print(f"MAE  : {mae:.3f} °C")
print(f"R²   : {r2:.3f}")

Evaluation Metrics
RMSE : 1.777 °C
MAE  : 1.433 °C
R²   : 0.902


## Get the prediction for Jan,25 (From Random Forest)

In [18]:
next_month_prediction = model_RF.predict(X_forecast)[0]

print("\nForecast for next month")
print(f"{next_month_prediction:.2f} °C")


Forecast for next month
4.40 °C
